# 07 · Recursion & Backtracking

**Recursion** = a function that calls itself on a smaller version of the problem.
**Backtracking** = recursion that *tries a choice, explores, then undoes the choice* to try the next one.

**The "pick / not pick" pattern** (subsequences & subsets): at every index we have exactly two choices:
1. **Pick** `nums[index]` → add it to the current subset, recurse on `index + 1`
2. **Not pick** → remove it again (the *backtrack* step), recurse on `index + 1`

This builds a recursion tree with `2ⁿ` leaves.

**Topics covered**
1. Generate all subsets (power set)
2. All subsequences with sum `k`
3. Print any one subsequence with sum `k`
4. Count subsequences with sum `k`
5. Binary strings of length `n` with no two consecutive 1s

## 1. All subsets (power set)

Two ways to generate them:
- **Bitmask** (from the Bit Manipulation notebook): each number `0 … 2ⁿ−1` is a mask.
- **Recursion:** pick / not pick.

**Time:** O(n · 2ⁿ) — there are 2ⁿ subsets and copying each one costs up to n.

In [1]:
# Method 1: bitmask
nums = [1, 2, 3]
n = len(nums)
result = []
for mask in range(1 << n):
    subset = [nums[i] for i in range(n) if mask & (1 << i)]
    result.append(subset)
print("Bitmask  :", result)

Bitmask  : [[], [1], [2], [1, 2], [3], [1, 3], [2, 3], [1, 2, 3]]


In [2]:
# Method 2: recursion (pick / not pick)
def generate_subsets(index, subset, nums, result):
    if index >= len(nums):                    # base case: decided about every element
        result.append(subset.copy())          # copy! otherwise later pops would modify what we stored
        return
    subset.append(nums[index])                # PICK
    generate_subsets(index + 1, subset, nums, result)
    subset.pop()                              # BACKTRACK: undo the pick
    generate_subsets(index + 1, subset, nums, result)   # NOT PICK

result = []
generate_subsets(0, [], [1, 2, 3], result)
print("Recursion:", result)

Recursion: [[1, 2, 3], [1, 2], [1, 3], [1], [2, 3], [2], [3], []]


> 💡 **Why `.copy()`?** `subset` is one list that is reused and modified through the whole recursion. Appending `subset` itself would store a *reference*, and every stored subset would end up empty once the recursion finishes.

## 2. All subsequences with sum `k`

Same pick / not-pick tree, but we carry a running `total`:
- `total == k` → record the subset and stop
- `total > k` → **prune** (stop early; works because all numbers are non-negative)
- `index` past the end → stop

Example: `nums = [2, 3, 6, 4, 7]`, `k = 7` → `[3, 4]` and `[7]`.

> ⚠️ The `total > k` pruning is valid only for **non-negative** numbers. With negatives (or zeros that could extend a valid subset), remove the early returns and check `total == k` only when `index` reaches the end.

In [3]:
def subsequences_with_sum(index, total, subset, nums, k, result):
    if total == k:
        result.append(subset.copy())
        return
    if total > k or index >= len(nums):       # prune / ran out of elements
        return
    subset.append(nums[index])                # pick
    subsequences_with_sum(index + 1, total + nums[index], subset, nums, k, result)
    subset.pop()                              # backtrack
    subsequences_with_sum(index + 1, total, subset, nums, k, result)   # not pick

result = []
subsequences_with_sum(0, 0, [], [2, 3, 6, 4, 7], 7, result)
print(result)                                 # [[3, 4], [7]]

[[3, 4], [7]]


*(The earlier version also subtracted from a local variable `sum` after popping, but never used it. That line was unnecessary, and the name `sum` hides Python's built-in `sum()`; both are removed here.)*

## 3. Print any ONE subsequence with sum `k`

Once we find one answer there is no reason to keep exploring. Make the function **return `True`** when it finds a solution and **`False`** otherwise, and stop immediately when a recursive call returns `True`.

This "early exit" pattern appears in many backtracking problems (N-Queens, Sudoku).

In [4]:
def one_subsequence_with_sum(index, total, subset, nums, k, result):
    if total == k:
        result.append(subset.copy())
        return True                            # found one → tell the caller to stop
    if total > k or index >= len(nums):
        return False
    subset.append(nums[index])                 # pick
    if one_subsequence_with_sum(index + 1, total + nums[index], subset, nums, k, result):
        return True                            # propagate success upward
    subset.pop()                               # backtrack
    return one_subsequence_with_sum(index + 1, total, subset, nums, k, result)  # not pick

result = []
found = one_subsequence_with_sum(0, 0, [], [2, 3, 6, 4, 7], 11, result)
print(found, result)                           # True [[2, 3, 6]]

True [[2, 3, 6]]


## 4. Count subsequences with sum `k`

Instead of storing subsets, **return a number**: the answer for a state is `pick_count + not_pick_count`.

- sum reached → `1` way
- sum exceeded / out of elements → `0` ways

Because we don't store lists, this version uses O(n) stack space only.

In [5]:
def count_subsequences(index, total, nums, k):
    if total == k:
        return 1
    if total > k or index >= len(nums):
        return 0
    pick = count_subsequences(index + 1, total + nums[index], nums, k)
    not_pick = count_subsequences(index + 1, total, nums, k)
    return pick + not_pick

print(count_subsequences(0, 0, [2, 3, 6, 4, 7], 9))   # 3 → [2,3,4], [2,7], [3,6]

3


## 5. Binary strings of length `n` with no consecutive 1s

Build the string one position at a time. At each position:
- We can **always place `0`**.
- We can place `1` **only if the previous character was not `1`**. We carry a flag `can_place_one` for this.

After placing `1`, the next call gets `can_place_one = False`; after `0`, the next call gets `True`.

For `n = 4` the answer has 8 strings (the count follows the Fibonacci sequence).

In [6]:
class Solution:
    def solve(self, index, can_place_one, chars, result):
        if index >= len(chars):                        # string is complete
            result.append("".join(chars))
            return
        chars[index] = "0"                             # option 1: place 0 (always allowed)
        self.solve(index + 1, True, chars, result)
        if can_place_one:                              # option 2: place 1 (only after a 0)
            chars[index] = "1"
            self.solve(index + 1, False, chars, result)
            chars[index] = "0"                         # backtrack

    def generate_binary_strings(self, n):
        chars = ["0"] * n
        result = []
        self.solve(0, True, chars, result)
        return result

print(Solution().generate_binary_strings(4))
print(len(Solution().generate_binary_strings(5)), "strings for n = 5")   # 13

['0000', '0001', '0010', '0100', '0101', '1000', '1001', '1010']
13 strings for n = 5


---
### ✅ Key takeaways
- **Pick / not pick** is the template for subsets, subsequences and combinations.
- **Backtrack** = undo your choice (`pop`) before trying the alternative.
- Store `subset.copy()`, not `subset`.
- Return `True/False` for "find one", return a **number** for "count", append to a **list** for "find all".
- Prune early (`total > k`) when the problem allows it.